In [34]:
from __future__ import annotations
import logging
from typing import List

from sqlalchemy import Column, Integer, Text, func 
from sqlalchemy.orm import declarative_base # type: ignore

from app.db.sqlalchemy import get_session_local
from app.services.llm_gateway import get_access_token, get_embeddings
from app.services.vector_store import fetch_top_k

CLIENT_ID: 5dc5f69a-b75b-42f7-baf1-b4df23f759f0


In [33]:
import os
import sys

# Force DB_HOST to localhost for local development
os.environ['DB_HOST'] = 'localhost'

# Clear all app-related modules to force reimport
modules_to_clear = [key for key in sys.modules.keys() if key.startswith('app')]
for module in modules_to_clear:
    del sys.modules[module]

In [35]:
Base = declarative_base()
SessionLocal = get_session_local()


class ChunkEmbedding(Base):
    __tablename__ = "chunk_embeddings"

    id = Column(Integer, primary_key=True)
    text = Column("chunk_text", Text, nullable=True)
    filename = Column(Text, nullable=True)

In [36]:
def _sanitize_tsquery(user_query: str) -> str:
    terms = [query.strip() for query in user_query.replace("\n", " ").split() if query.strip()]
    escaped = []
    for term in terms:
        safe = "".join(character for character in term if character.isalnum() or character in {"_", "-"})
        if safe:
            escaped.append(f"{safe}:*")
    return " & ".join(escaped) if escaped else ""

In [41]:
def _full_text_search(*, user_query: str, limit: int, file_filter: str | None = None) -> List[dict]:
    textsearch_query = _sanitize_tsquery(user_query)
    if not textsearch_query:
         return []

    db = SessionLocal()
    try:
        query = db.query(
            ChunkEmbedding,
            func.ts_rank_cd(
                func.to_tsvector("english", ChunkEmbedding.text),
                func.to_tsquery("english", textsearch_query),
            ).label("tsv_score"),
        ).filter(
            func.to_tsvector("english", ChunkEmbedding.text).op("@@")(
                func.to_tsquery("english", textsearch_query)
            )
        )

        if file_filter:
            query = query.filter(ChunkEmbedding.filename == file_filter)


        results = (
            query.order_by(
                func.ts_rank_cd(
                    func.to_tsvector("english", ChunkEmbedding.text),
                    func.to_tsquery("english", textsearch_query),
                ).desc()
            )
            .limit(limit)
            .all()
        )

        out: List[dict] = []
        for emb_row, tsv_score in results:
            out.append(
                {
                    "id": int(emb_row.id),
                    "chunk_text": emb_row.text,
                    "filename": emb_row.filename,
                    "lexical_score": float(tsv_score) if tsv_score is not None else 0.0,
                }
            )
        return out
    except Exception as e:
        return []
    finally:
        db.close()

In [38]:
def _rrf_score(*, rank: int | None, k: int) -> float:
    """Reciprocal Rank Fusion contribution for a single list.

    rank is 1-based (1 is best). If rank is None (doc not in list) -> 0.
    """

    if rank is None or rank <= 0:
        return 0.0
    return 1.0 / float(k + rank)

In [46]:
def hybrid_retrieve(
    *,
    query: str,
    limit: int = 5,
    alpha: float = 0.6,
    file_filter: str | None = None,
    mode: str = "rrf",
    rrf_k: int = 60,
) -> dict:
    try:
        access_token = get_access_token()
        query_embedding = get_embeddings([query], access_token)[0]["embedding"]
    except Exception as e:
        raise
 
    candidate_k = max(limit * 3, 10)
    try:
        vector_candidates = fetch_top_k(query_embedding, top_k=candidate_k, filename=file_filter)
        print(f"count of vector search results: {len(vector_candidates)}")
    except Exception as e:
        raise
   
    tsv_rows = _full_text_search(user_query=query, limit=candidate_k, file_filter=file_filter)  ##text search results
    print(f"count of full text search results: {len(tsv_rows)}")

    mode = (mode or "blend").lower().strip()
    if mode not in {"blend", "rrf"}:
        raise ValueError("mode must be one of: blend, rrf")
    if rrf_k <= 0:
        raise ValueError("rrf_k must be > 0")

    if mode == "rrf":
        # Rank maps (1-based)
        vector_rank: dict[str, int] = {}
        for i, (chunk_text, _dist) in enumerate(vector_candidates, start=1):
            if not chunk_text:
                continue
            vector_rank.setdefault(chunk_text, i)

        lexical_rank: dict[str, int] = {}
        filename_map: dict[str, str] = {}
        chunk_id_map: dict[str, int] = {}

        for i, row in enumerate(tsv_rows, start=1):
            chunk_text = row.get("chunk_text")
            if not chunk_text:
                continue
            lexical_rank.setdefault(chunk_text, i)
            if row.get("filename"):
                filename_map.setdefault(chunk_text, row.get("filename"))
            if row.get("id"):
                chunk_id_map.setdefault(chunk_text, row.get("id"))

        # Union of candidates
        merged: dict[str, dict] = {}
        for chunk_text_vector, rank_vector in vector_rank.items():
            merged.setdefault(chunk_text_vector, {"chunk_text": chunk_text_vector, "filename": None, "chunk_id": None})
            merged[chunk_text_vector]["vector_rank"] = rank_vector

        for chunk_text_lexical, rank_lexical in lexical_rank.items():
            merged.setdefault(chunk_text_lexical, {"chunk_text": chunk_text_lexical, "filename": None, "chunk_id": None})
            merged[chunk_text_lexical]["lexical_rank"] = rank_lexical
            if chunk_text_lexical in filename_map and not merged[chunk_text_lexical].get("filename"):
                merged[chunk_text_lexical]["filename"] = filename_map[chunk_text_lexical]
            if chunk_text_lexical in chunk_id_map and not merged[chunk_text_lexical].get("chunk_id"):
                merged[chunk_text_lexical]["chunk_id"] = chunk_id_map[chunk_text_lexical]

        items = list(merged.values())
        for item in items:
            vector_rank = item.get("vector_rank")
            lexical_rank = item.get("lexical_rank")
            item["rrf_score"] = (float(alpha) * _rrf_score(rank=vector_rank, k=int(rrf_k))) + (
                (1.0 - float(alpha)) * _rrf_score(rank=lexical_rank, k=int(rrf_k))
            )

        items.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
        items = items[:limit]

        # Tighten filename and chunk_id consistency for top results.
        top_texts = [item.get("chunk_text") for item in items if item.get("chunk_text")]
        if top_texts:
            db = SessionLocal()
            try:
                query = db.query(ChunkEmbedding.id, ChunkEmbedding.text, ChunkEmbedding.filename).filter(ChunkEmbedding.text.in_(top_texts))
                if file_filter:
                    query = query.filter(ChunkEmbedding.filename == file_filter)
                chunk_info = {chunk_text: (chunk_id, fn) for chunk_id, chunk_text, fn in query.all() if chunk_text}
                for item in items:
                    if item.get("chunk_text") in chunk_info:
                        chunk_id, filename = chunk_info[item.get("chunk_text")]
                        item["chunk_id"] = chunk_id
                        item["filename"] = filename
            finally:
                db.close()
 
        return {
            "results": items,
            "params": {
                "limit": limit,
                "alpha": alpha,
                "file_filter": file_filter,
                "fusion_mode": "rrf",
                "rrf_k": int(rrf_k),
            },
        }

In [47]:
hybrid_retrieve(
    query="hi",
    limit=5,
    alpha=0.6,
    file_filter=None,
    mode="rrf",
    rrf_k=60,
)

count of vector search results: 15
count of full text search results: 15


{'results': [{'chunk_text': '\nFigure 1 — Screening, Randomization, and Flow of Patients in the SURPASS-5 Trial\na Includes 1 participant each with absence of type 2 diabetes diagnosis, body mass index <23, serum calcitonin level 35 ng/L, not able to meet study requirements or of completing protocol due to drug/alcohol misuse or psychiatric disorder as judged by the investigator, and history of active/untreated malignancy or in remission.\n\nb See eTable 3 in Supplement 2 for the details of the adverse events that led to study drug discontinuation.\n\nc Study discontinuations include both before and after the primary end point visit.\n\n586 Adults with type 2 diabetes receiving stable doses of insulin glargine assessed for eligibility\n\n111 Excluded\n\n93 Did not meet screening criteria\n\n52 Did not meet baseline HbA1c criteria\n\n12 History of diabetic retinopathy or maculopathy\n\n10 Did not meet baseline eGFR criteria\n\n5 Were not receiving stable daily insulin glargine dose\n\n4